<a href="https://colab.research.google.com/github/raquelsantos81/cbf_2015_pdfs/blob/main/Borderos_2015_Colab_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Extração dos borderôs do Brasileirão 2015 (Série A)
Lê cada PDF com um modelo de visão (Gemini), grava o resultado e **confere as contas**:
soma dos setores = receita bruta declarada; soma dos vendidos = total declarado; quantidade × preço = arrecadação.
O que não fechar vai para a lista **REVISAR** — esses jogos precisam de olho humano.

**Como usar:** rode as células na ordem. Se a cota gratuita acabar, o script para sozinho:
é só rodar a célula 4 de novo no dia seguinte (ele retoma de onde parou).

## Célula 1 — instalar e baixar os PDFs

In [1]:
!pip -q install google-genai pymupdf
!git clone https://github.com/raquelsantos81/cbf_2015_pdfs.git /content/cbf_2015_pdfs

Cloning into '/content/cbf_2015_pdfs'...
remote: Enumerating objects: 385, done.
remote: Total 385 (delta 0), reused 0 (delta 0), pack-reused 385 (from 1)
Receiving objects: 100% (385/385), 259.89 MiB | 30.08 MiB/s, done.
Updating files: 100% (383/383), done.


## Célula 2 — (recomendado) guardar o progresso no Google Drive
Assim, se o Colab desconectar, você não perde nada.

In [2]:
from google.colab import drive
drive.mount('/content/drive')
SAIDA = '/content/drive/MyDrive/borderos_2015_saida'

Mounted at /content/drive


## Célula 3 — configuração e funções (só rodar, não precisa mexer)

In [3]:
import os, io, re, json, time, glob, csv, getpass

PASTA_PDF = "/content/cbf_2015_pdfs"
SAIDA = globals().get("SAIDA", "/content/saida")
MODELO = "gemini-3.1-flash-lite"   # rápido (~5 s/jogo nos testes). Se der erro 404, rode listar_modelos() e troque
DPI = 130                     # resolução das imagens enviadas
PAUSA_S = 7                   # segundos entre jogos (protege a cota gratuita)
PULAR_FEITOS = True           # pula os 16 jogos que já foram lidos e validados manualmente
JA_FEITOS = {1, 2, 3, 4, 37, 60, 87, 169, 173, 191, 217, 239, 259, 269, 343, 379}
os.makedirs(SAIDA, exist_ok=True)
ARQ_PROGRESSO = os.path.join(SAIDA, "progresso.jsonl")

PROMPT = """Você está lendo imagens de UM borderô (boletim financeiro) de um jogo do Campeonato Brasileiro Série A 2015.
O documento pode ser impresso, escaneado ou MANUSCRITO. Pode ter 1 a 4 páginas; a tabela de ingressos/receitas costuma estar na primeira.

REGRAS:
- Transcreva EXATAMENTE o que está escrito. NÃO corrija, NÃO estime, NÃO complete, NÃO recalcule.
- Se um valor estiver ilegível ou em branco, use null. Nunca invente.
- Cada linha de setor da tabela de ingressos vira um item em "setores", na ordem em que aparece.
  Atenção: às vezes os nomes dos setores ficam desalinhados das linhas de números; use a ordem das linhas.
- NÃO inclua linhas de "Sub-Total", "Total" ou "Totais" como setores: elas só entram em "totais_declarados" (se forem o total geral).
- Linhas de "público não pagante", cortesias, gratuidades, sócios e credenciais também entram em "setores".
- Números em formato brasileiro ("1.234,50") devem sair como número com ponto decimal (1234.50).
- Ignore as despesas (B1, B2, B3), retenções e divisão da renda. Só ingressos e totais da tabela de receitas.

Responda SOMENTE com JSON neste formato:
{
 "data": "AAAA-MM-DD" ou null,
 "mandante": texto ou null,
 "visitante": texto ou null,
 "estadio": texto ou null,
 "cidade_uf": texto ou null,
 "numero_bf": texto ou null,
 "setores": [
   {"setor": texto, "a_venda": numero ou null, "devolvidos": numero ou null, "vendidos": numero ou null,
    "preco_unitario": numero ou null, "arrecadacao": numero ou null}
 ],
 "totais_declarados": {"a_venda": numero ou null, "devolvidos": numero ou null,
                       "vendidos": numero ou null, "receita_bruta": numero ou null},
 "observacoes": texto curto (ex.: "manuscrito", "linha ilegível", "carimbo sobre números") ou null
}"""


def numero_do_arquivo(caminho):
    """142 + N  ->  N (jogo 1..380)."""
    base = os.path.basename(caminho)[:-5]  # tira 'b.pdf'
    return int(base[3:]) if base.startswith("142") else None


def lista_arquivos():
    arqs = [f for f in glob.glob(os.path.join(PASTA_PDF, "142*b.pdf"))]
    arqs = [f for f in arqs if numero_do_arquivo(f) is not None]
    return sorted(arqs, key=numero_do_arquivo)


def renderizar(caminho, dpi=None):
    """Devolve lista de PNG (bytes), uma por página."""
    try:
        import pymupdf as fitz
    except ImportError:
        import fitz
    doc = fitz.open(caminho)
    out = []
    for pg in doc:
        pix = pg.get_pixmap(dpi=dpi or DPI)
        out.append(pix.tobytes("png"))
    return out


def num(x):
    """Converte para float aceitando texto brasileiro; None se não der."""
    if x is None or x == "":
        return None
    if isinstance(x, (int, float)):
        return float(x)
    s = str(x).strip().replace("R$", "").replace(" ", "")
    if re.fullmatch(r"-?\d{1,3}(\.\d{3})*(,\d+)?", s):
        s = s.replace(".", "").replace(",", ".")
    elif "," in s and "." not in s:
        s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return None


PADRAO_SUBTOTAL = re.compile(r"(sub[\s\-\.]*total|^\s*totais?\b)", re.I)


def limpar_subtotais(d):
    """Tira das linhas de setor as linhas de 'Sub-Total'/'Total' que o modelo leu como se fossem setores."""
    manter, removidas = [], list(d.get("linhas_removidas") or [])
    for x in d.get("setores") or []:
        if PADRAO_SUBTOTAL.search(str(x.get("setor") or "")):
            removidas.append(x)
        else:
            manter.append(x)
    d["setores"] = manter
    if removidas:
        d["linhas_removidas"] = removidas
    return d


def validar(d):
    """Confere as contas. Devolve (status, lista_de_problemas)."""
    probs = []
    S = d.get("setores") or []
    if not S:
        return "REVISAR", ["nenhum setor lido"]
    T = d.get("totais_declarados") or {}
    soma_ar = sum(num(x.get("arrecadacao")) or 0 for x in S)
    rb, tv = num(T.get("receita_bruta")), num(T.get("vendidos"))
    if rb is None:
        probs.append("receita bruta declarada não lida")
    elif abs(soma_ar - rb) > 0.5:
        probs.append(f"soma das arrecadações {soma_ar:,.2f} ≠ receita bruta {rb:,.2f}")
    v_todos = sum(num(x.get("vendidos")) or 0 for x in S)
    v_pagantes = sum(num(x.get("vendidos")) or 0 for x in S if (num(x.get("arrecadacao")) or 0) > 0)
    if tv is None:
        probs.append("total de vendidos declarado não lido")
    elif min(abs(v_todos - tv), abs(v_pagantes - tv)) > 0.5:
        probs.append(f"vendidos: soma {v_todos:,.0f} (só pagantes {v_pagantes:,.0f}) ≠ total {tv:,.0f}")
    for x in S:
        p, v, a = num(x.get("preco_unitario")), num(x.get("vendidos")), num(x.get("arrecadacao"))
        nome = x.get("setor") or "(sem nome)"
        # célula em branco numa linha sem movimento (vendidos e arrecadação vazios/zero) = zero, não é erro de leitura
        if v is None and (a or 0) > 0:
            probs.append(f"'{nome}': vendidos ausente, mas há arrecadação {a:,.2f}")
        if p and p > 0 and a is None and (v or 0) > 0:
            probs.append(f"'{nome}': arrecadação ausente com {v:,.0f} vendidos")
        if p and p > 0 and v is not None and a is not None and abs(p * v - a) > max(0.5, 0.01 * a):
            probs.append(f"'{nome}': {v:,.0f} × {p:,.2f} ≠ {a:,.2f}")
    return ("OK" if not probs else "REVISAR"), probs


def revalidar():
    """Limpa subtotais e reaplica a conferência aos jogos já salvos, SEM chamar o modelo de novo."""
    feitos = carregar_progresso()
    antes = sum(1 for d in feitos.values() if d["status"] == "OK")
    with open(ARQ_PROGRESSO, "w", encoding="utf8") as fh:
        for n in sorted(feitos):
            d = limpar_subtotais(feitos[n])
            d["status"], d["problemas"] = validar(d)
            fh.write(json.dumps(d, ensure_ascii=False) + "\n")
    ok = sum(1 for d in feitos.values() if d["status"] == "OK")
    print(f"Jogos: {len(feitos)} | OK antes: {antes} -> agora: {ok} | REVISAR: {len(feitos) - ok}")


def limpar_json(txt):
    txt = txt.strip()
    txt = re.sub(r"^```(?:json)?|```$", "", txt, flags=re.M).strip()
    return json.loads(txt)


def chamar_modelo(imagens):
    """Envia as páginas ao Gemini e devolve o texto da resposta."""
    from google import genai
    from google.genai import types
    global _CLIENT
    if "_CLIENT" not in globals() or _CLIENT is None:
        chave = os.environ.get("GEMINI_API_KEY") or getpass.getpass("Cole sua chave do Google AI Studio: ")
        _CLIENT = genai.Client(api_key=chave)
    partes = [types.Part.from_bytes(data=b, mime_type="image/png") for b in imagens]
    r = _CLIENT.models.generate_content(
        model=MODELO,
        contents=[PROMPT] + partes,
        config=types.GenerateContentConfig(response_mime_type="application/json", temperature=0),
    )
    return r.text


def listar_modelos():
    """Mostra os modelos que a SUA chave pode usar para ler imagens. Copie um nome 'flash' para a variável MODELO."""
    from google import genai
    global _CLIENT
    if "_CLIENT" not in globals() or _CLIENT is None:
        chave = os.environ.get("GEMINI_API_KEY") or getpass.getpass("Cole sua chave do Google AI Studio: ")
        _CLIENT = genai.Client(api_key=chave)
    for m in _CLIENT.models.list():
        acoes = getattr(m, "supported_actions", None) or []
        if "generateContent" in acoes:
            print(m.name.replace("models/", ""))


def carregar_progresso():
    feitos = {}
    if os.path.exists(ARQ_PROGRESSO):
        for l in open(ARQ_PROGRESSO, encoding="utf8"):
            l = l.strip()
            if l:
                d = json.loads(l)
                feitos[d["n"]] = d
    return feitos


def processar(limite=None, so_revisar=False):
    """Lê os jogos que faltam. Pode ser interrompido e rodado de novo."""
    feitos = carregar_progresso()
    n_proc = 0
    for f in lista_arquivos():
        n = numero_do_arquivo(f)
        if PULAR_FEITOS and n in JA_FEITOS:
            continue
        if n in feitos and not (so_revisar and feitos[n]["status"] != "OK"):
            continue
        if so_revisar and n not in feitos:
            continue
        if limite and n_proc >= limite:
            break
        d, erro = None, None
        for tent in range(1, 6):
            try:
                d = limpar_json(chamar_modelo(renderizar(f)))
                break
            except Exception as e:
                erro = str(e)
                low = erro.lower()
                if "404" in erro or "not_found" in low or "not found" in low:
                    print("\nO MODELO '" + MODELO + "' não está disponível para a sua chave. Nada foi gravado.")
                    print("Rode listar_modelos(), escolha um nome 'flash' da lista e troque a variável MODELO.")
                    return
                if "per day" in low or "daily" in low:
                    print("\nA cota DIÁRIA acabou. Rode a célula de novo amanhã: o progresso está salvo.")
                    return
                espera = 60 * tent if ("429" in erro or "quota" in low or "exhausted" in low) else 10 * tent
                print(f"  jogo {n}: tentativa {tent} falhou ({erro[:90]}). Esperando {espera}s...")
                time.sleep(espera)
        if d is None:
            d = {"setores": [], "observacoes": f"falha ao ler: {erro}"}
        d["n"], d["arquivo"] = n, os.path.basename(f)
        limpar_subtotais(d)
        d["status"], d["problemas"] = validar(d)
        with open(ARQ_PROGRESSO, "a", encoding="utf8") as fh:
            fh.write(json.dumps(d, ensure_ascii=False) + "\n")
        n_proc += 1
        print(f"jogo {n:>3}  {d['status']:<8} {d.get('mandante')} x {d.get('visitante')}  {('; '.join(d['problemas']))[:90]}")
        time.sleep(PAUSA_S)
    print("\nFim desta rodada.")


def br(x):
    v = num(x)
    return "" if v is None else f"{v:.2f}".replace(".", ",")


def exportar():
    """Gera os dois CSVs (abrem direto no Excel brasileiro)."""
    # a última leitura de cada jogo vale
    feitos = carregar_progresso()
    p1 = os.path.join(SAIDA, "Borderos_2015_setores_lidos_pela_IA.csv")
    p2 = os.path.join(SAIDA, "PARA_REVISAR.csv")
    with open(p1, "w", newline="", encoding="utf-8-sig") as f1, open(p2, "w", newline="", encoding="utf-8-sig") as f2:
        w1, w2 = csv.writer(f1, delimiter=";"), csv.writer(f2, delimiter=";")
        w1.writerow(["N_arquivo", "Data", "Jogo", "Estadio", "Setor", "A_venda", "Devolvidos", "Vendidos",
                     "Preco_unitario", "Arrecadacao", "Conferencia"])
        w2.writerow(["N_arquivo", "Arquivo", "Jogo", "Problemas", "Observacoes"])
        for n in sorted(feitos):
            d = feitos[n]
            jogo = f"{d.get('mandante')} x {d.get('visitante')}"
            for x in d.get("setores", []):
                w1.writerow([n, d.get("data"), jogo, d.get("estadio"), x.get("setor"),
                             "" if num(x.get("a_venda")) is None else int(num(x["a_venda"])),
                             "" if num(x.get("devolvidos")) is None else int(num(x["devolvidos"])),
                             "" if num(x.get("vendidos")) is None else int(num(x["vendidos"])),
                             br(x.get("preco_unitario")), br(x.get("arrecadacao")), d["status"]])
            if d["status"] != "OK":
                w2.writerow([n, d["arquivo"], jogo, " | ".join(d["problemas"]), d.get("observacoes")])
    ok = sum(1 for d in feitos.values() if d["status"] == "OK")
    print(f"Jogos lidos: {len(feitos)} | conferidos (OK): {ok} | para revisar: {len(feitos) - ok}")
    print("Arquivos:", p1, "e", p2)

In [4]:
print(os.path.exists(ARQ_PROGRESSO), len(carregar_progresso()))

True 364


In [5]:
revalidar()

Jogos: 364 | OK antes: 251 -> agora: 326 | REVISAR: 38


In [6]:
exportar()

Jogos lidos: 364 | conferidos (OK): 326 | para revisar: 38
Arquivos: /content/drive/MyDrive/borderos_2015_saida/Borderos_2015_setores_lidos_pela_IA.csv e /content/drive/MyDrive/borderos_2015_saida/PARA_REVISAR.csv


## Célula 4 — rodar a extração
Teste primeiro com `processar(limite=3)`. Se os 3 vierem OK (ou com problemas coerentes), rode `processar()` para o resto.

In [ ]:
processar(limite=3)

In [ ]:
processar()

## Célula 5 — gerar as planilhas
Baixe os dois arquivos da pasta de saída. Os jogos listados em PARA_REVISAR devem ser conferidos
(você pode me mandar o número do arquivo e eu leio a imagem).

In [7]:
exportar()

Jogos lidos: 364 | conferidos (OK): 326 | para revisar: 38
Arquivos: /content/drive/MyDrive/borderos_2015_saida/Borderos_2015_setores_lidos_pela_IA.csv e /content/drive/MyDrive/borderos_2015_saida/PARA_REVISAR.csv
